# Topic 8: Searching Algorithms (Binary Search)

**Goal**: Master binary search and its many variations — the most versatile O(log n) technique.  
**Time**: ~4-5 hours  
**Prereqs**: Topics 0-2, 7

---

## Why Binary Search?

Can search **1 billion** items in ~**30 steps**.

The key insight: if the search space is **sorted** (or has a **monotonic property**), you can eliminate half of it each step.

```
Linear Search vs Binary Search — searching 1,000,000,000 items:

  Linear:  O(n)       = 1,000,000,000 steps  (worst case)
  Binary:  O(log n)   =            30 steps   (always)

  ┌──────────────────────────────────────────────────┐
  │ n             │ Linear (worst) │ Binary (worst)  │
  ├──────────────────────────────────────────────────┤
  │ 100           │           100  │              7  │
  │ 10,000        │        10,000  │             14  │
  │ 1,000,000     │     1,000,000  │             20  │
  │ 1,000,000,000 │ 1,000,000,000  │             30  │
  └──────────────────────────────────────────────────┘
```

Binary search applies far beyond "find a number in a sorted array." It shows up in:

| Pattern | Example |
|---|---|
| Exact value in sorted data | Classic binary search |
| First/last occurrence | Find boundaries |
| Rotated sorted arrays | Modified comparison logic |
| Peak finding | Compare neighbors |
| Search on answer space | "What's the minimum X such that..." |

---

## The Standard Template

The classic binary search: find `target` in a sorted array.

```
[1, 3, 5, 7, 9, 11, 13, 15], target = 7
 L              M              R
 [1, 3, 5, 7]  ← go left (7 < 9)
  L     M    R
     [5, 7]  ← go right (7 > 5)
      L=M=R
       [7]   ← found!
```

**The core idea at every step:**

```
  ┌──────────────────────────────────────────────┐
  │  L          search space          R          │
  │  ├──────────────┬──────────────────┤         │
  │                 M                            │
  │                                              │
  │  if target < arr[M]:  discard right half     │
  │  if target > arr[M]:  discard left half      │
  │  if target == arr[M]: done!                  │
  └──────────────────────────────────────────────┘
```

Each step cuts the search space in half → **O(log n)**.

In [ ]:
def binary_search(nums, target):
    left, right = 0, len(nums) - 1
    step = 0

    while left <= right:
        mid = left + (right - left) // 2
        step += 1
        print(f"  Step {step}: left={left} right={right} mid={mid} | nums[{mid}]={nums[mid]}", end="")

        if nums[mid] == target:
            print(f"  == {target} ✓ FOUND")
            return mid
        elif nums[mid] < target:
            print(f"  < {target} → search right")
            left = mid + 1
        else:
            print(f"  > {target} → search left")
            right = mid - 1

    print(f"  Step {step + 1}: left={left} > right={right} → NOT FOUND")
    return -1


print("=== Standard Binary Search ===")
nums = [1, 3, 5, 7, 9, 11, 13, 15]

print(f"\nArray: {nums}")
print(f"Search for 7:")
result = binary_search(nums, 7)
print(f"→ Index: {result}\n")

print(f"Search for 12:")
result = binary_search(nums, 12)
print(f"→ Index: {result}\n")

print(f"Search for 1 (first element):")
result = binary_search(nums, 1)
print(f"→ Index: {result}\n")

print(f"Search for 15 (last element):")
result = binary_search(nums, 15)
print(f"→ Index: {result}")

---

### The Three Binary Search Templates

Different loop conditions suit different problems. Know all three:

```
┌────────────────────────────────────────────────────────────────────────┐
│  TEMPLATE 1: while left <= right                                     │
│  ─────────────────────────────────                                   │
│  • Search space: [left, right] (inclusive both ends)                  │
│  • Terminates when: left > right (space is empty)                    │
│  • Use for: exact match, standard search                            │
│  • After loop: element NOT found                                     │
│                                                                      │
│  left, right = 0, n-1                                                │
│  while left <= right:                                                │
│      mid = left + (right - left) // 2                                │
│      if found: return mid                                            │
│      elif go_right: left = mid + 1                                   │
│      else: right = mid - 1                                           │
├────────────────────────────────────────────────────────────────────────┤
│  TEMPLATE 2: while left < right                                      │
│  ────────────────────────────────                                    │
│  • Search space: [left, right) or [left, right]                      │
│  • Terminates when: left == right (one element left)                 │
│  • Use for: find boundary (first true, last true)                    │
│  • After loop: left == right is the answer candidate                 │
│                                                                      │
│  left, right = 0, n-1   (or n for upper bound)                       │
│  while left < right:                                                 │
│      mid = left + (right - left) // 2                                │
│      if condition(mid): right = mid      ← mid could be answer       │
│      else: left = mid + 1                                            │
│  return left                                                         │
├────────────────────────────────────────────────────────────────────────┤
│  TEMPLATE 3: while left < right - 1                                  │
│  ──────────────────────────────────                                  │
│  • Search space: stops when left and right are neighbors             │
│  • Terminates when: left + 1 == right                                │
│  • Use for: need to compare two adjacent candidates                  │
│  • After loop: check BOTH left and right                             │
│                                                                      │
│  left, right = 0, n-1                                                │
│  while left < right - 1:                                             │
│      mid = left + (right - left) // 2                                │
│      if condition: right = mid                                       │
│      else: left = mid                                                │
│  # check both nums[left] and nums[right]                            │
└────────────────────────────────────────────────────────────────────────┘
```

**Rule of thumb:**
- Need exact match? → Template 1 (`<=`)
- Need a boundary (first/last position)? → Template 2 (`<`)
- Need to compare two adjacent elements? → Template 3 (`< right - 1`)

**Why `mid = left + (right - left) // 2` instead of `(left + right) // 2`?**  
Prevents integer overflow in languages like C++/Java. In Python it doesn't matter, but it's good habit.

---

## Problem 1: Find First and Last Position of Element in Sorted Array (LC #34)

Given a sorted array, find the starting and ending position of a target value.  
Return `[-1, -1]` if not found. Must be **O(log n)**.

**Strategy**: Use binary search **twice** — once to find the leftmost occurrence, once for the rightmost.

```
nums = [5, 7, 7, 8, 8, 8, 10], target = 8
                  ^     ^
                first  last

Finding FIRST 8:                    Finding LAST 8:
  Found 8 at mid? Don't stop!         Found 8 at mid? Don't stop!
  Keep searching LEFT (right=mid-1)   Keep searching RIGHT (left=mid+1)
  Record mid as candidate             Record mid as candidate

  ┌───────────────────────────────┐
  │ Key insight: when you find     │
  │ the target, DON'T return.     │
  │ Keep going to find the        │
  │ boundary.                     │
  └───────────────────────────────┘
```

In [ ]:
def search_range(nums, target):
    def find_left():
        left, right = 0, len(nums) - 1
        result = -1
        while left <= right:
            mid = left + (right - left) // 2
            print(f"    L={left} R={right} M={mid} | nums[{mid}]={nums[mid]}", end="")
            if nums[mid] == target:
                result = mid
                print(f"  == {target}, record {mid}, search LEFT")
                right = mid - 1
            elif nums[mid] < target:
                print(f"  < {target} → go right")
                left = mid + 1
            else:
                print(f"  > {target} → go left")
                right = mid - 1
        return result

    def find_right():
        left, right = 0, len(nums) - 1
        result = -1
        while left <= right:
            mid = left + (right - left) // 2
            print(f"    L={left} R={right} M={mid} | nums[{mid}]={nums[mid]}", end="")
            if nums[mid] == target:
                result = mid
                print(f"  == {target}, record {mid}, search RIGHT")
                left = mid + 1
            elif nums[mid] < target:
                print(f"  < {target} → go right")
                left = mid + 1
            else:
                print(f"  > {target} → go left")
                right = mid - 1
        return result

    print(f"  Finding FIRST {target}:")
    first = find_left()
    print(f"  → First at index {first}\n")

    print(f"  Finding LAST {target}:")
    last = find_right()
    print(f"  → Last at index {last}")

    return [first, last]


print("=== Find First and Last Position (LC #34) ===")

nums = [5, 7, 7, 8, 8, 8, 10]
print(f"\nArray: {nums}, target=8")
result = search_range(nums, 8)
print(f"Result: {result}\n")

print(f"Array: {nums}, target=6")
result = search_range(nums, 6)
print(f"Result: {result}\n")

nums2 = [1, 1, 1, 1, 1]
print(f"Array: {nums2}, target=1")
result = search_range(nums2, 1)
print(f"Result: {result}")

---

## Problem 2: Search in Rotated Sorted Array (LC #33)

A sorted array was **rotated** at some pivot. Search for a target in O(log n).

```
Original sorted: [0, 1, 2, 3, 4, 5, 6, 7]
Rotated at idx 4: [4, 5, 6, 7, 0, 1, 2, 3]
                            ↑ rotation point

[4, 5, 6, 7, 0, 1, 2, 3]
 ←─sorted──→  ←─sorted──→
 Left half     Right half
```

**Key insight**: Even though the whole array isn't sorted, **one half is ALWAYS sorted**.

```
  At each step, find the sorted half:

  [4, 5, 6, 7, 0, 1, 2, 3]    target = 1
   L           M           R
   
   nums[L]=4 <= nums[M]=7  → Left half [4,5,6,7] is sorted
   Is target in [4..7]?  NO (1 < 4)
   → Search right half

  [0, 1, 2, 3]
   L     M    R

   nums[L]=0 <= nums[M]=1  → Left half [0,1] is sorted
   Is target in [0..1]?  YES (0 <= 1 <= 1)
   → Search left half

  [0, 1]
   L=M  R
   nums[M]=0 != 1 → go right

  [1]
   L=M=R  → Found!
```

**Algorithm:**
1. Compute mid
2. Determine which half is sorted (compare `nums[left]` with `nums[mid]`)
3. Check if target is in the sorted half
4. If yes, search that half; if no, search the other half

In [ ]:
def search_rotated(nums, target):
    left, right = 0, len(nums) - 1

    while left <= right:
        mid = left + (right - left) // 2
        print(f"  L={left} R={right} M={mid} | nums[{mid}]={nums[mid]}", end="")

        if nums[mid] == target:
            print(f"  ✓ FOUND")
            return mid

        if nums[left] <= nums[mid]:
            print(f"  left half [{nums[left]}..{nums[mid]}] sorted", end="")
            if nums[left] <= target < nums[mid]:
                print(f" → target in left half")
                right = mid - 1
            else:
                print(f" → target NOT in left, go right")
                left = mid + 1
        else:
            print(f"  right half [{nums[mid]}..{nums[right]}] sorted", end="")
            if nums[mid] < target <= nums[right]:
                print(f" → target in right half")
                left = mid + 1
            else:
                print(f" → target NOT in right, go left")
                right = mid - 1

    print(f"  L={left} > R={right} → NOT FOUND")
    return -1


print("=== Search in Rotated Sorted Array (LC #33) ===")

nums = [4, 5, 6, 7, 0, 1, 2]
print(f"\nArray: {nums}")

print(f"\nSearch for 0:")
print(f"→ Index: {search_rotated(nums, 0)}\n")

print(f"Search for 5:")
print(f"→ Index: {search_rotated(nums, 5)}\n")

print(f"Search for 3:")
print(f"→ Index: {search_rotated(nums, 3)}")

---

## Problem 3: Find Minimum in Rotated Sorted Array (LC #153)

Find the minimum element in a rotated sorted array (no duplicates).

```
[4, 5, 6, 7, 0, 1, 2]
             ↑ minimum is at the rotation point

Key insight: compare nums[mid] with nums[right].

  nums[mid] > nums[right]  →  min is in right half
       (there's a drop somewhere to the right)

  nums[mid] <= nums[right] →  min is in left half (including mid)
       (right side is fine, drop must be to the left or AT mid)

  [4, 5, 6, 7, 0, 1, 2]
   L           M        R
   nums[3]=7 > nums[6]=2  → min is to the right
   
  [0, 1, 2]
   L  M    R
   nums[5]=1 <= nums[6]=2  → min is to the left (or at mid)
   
  [0, 1]
   L=M  R
   nums[4]=0 <= nums[5]=1  → min is to the left (or at mid)
   
  [0]
   L=R  → answer is 0
```

In [ ]:
def find_min_rotated(nums):
    left, right = 0, len(nums) - 1

    while left < right:
        mid = left + (right - left) // 2
        print(f"  L={left} R={right} M={mid} | nums[{mid}]={nums[mid]} vs nums[{right}]={nums[right]}", end="")

        if nums[mid] > nums[right]:
            print(f"  → min is RIGHT of mid")
            left = mid + 1
        else:
            print(f"  → min is LEFT of mid (or AT mid)")
            right = mid

    print(f"  L=R={left} → minimum = {nums[left]}")
    return nums[left]


print("=== Find Minimum in Rotated Sorted Array (LC #153) ===")

test_cases = [
    [4, 5, 6, 7, 0, 1, 2],
    [3, 4, 5, 1, 2],
    [11, 13, 15, 17],
    [2, 1],
]

for nums in test_cases:
    print(f"\nArray: {nums}")
    result = find_min_rotated(nums)
    print(f"→ Minimum: {result}")

---

## Problem 4: Find Peak Element (LC #162)

Find a **peak element** — an element greater than its neighbors.  
The array is NOT sorted. Return ANY peak index. Must be O(log n).

`nums[-1] = nums[n] = -∞` (boundaries are negative infinity).

```
Why does binary search work on an UNSORTED array?

  [1, 2, 1, 3, 5, 6, 4]
         ↑        ↑
        peak     peak

  Key insight: if nums[mid] < nums[mid + 1],
  then a peak MUST exist to the RIGHT.

  Why? The values are going UP at mid. Either:
  - They keep going up and hit the boundary (peak at end), OR
  - They come back down somewhere (peak in between)

  Either way, there's a peak on the right side!

  Visualized:
  
     6 ·              Imagine walking uphill.
     5 · ·            If you're going UP,
     4 ·   · ←peak   a peak is AHEAD.
     3 ·                  
     2 · ·                
     1 ·   · ←peak       
       0 1 2 3 4 5 6      
```

In [ ]:
def find_peak_element(nums):
    left, right = 0, len(nums) - 1

    while left < right:
        mid = left + (right - left) // 2
        print(f"  L={left} R={right} M={mid} | nums[{mid}]={nums[mid]} vs nums[{mid+1}]={nums[mid+1]}", end="")

        if nums[mid] < nums[mid + 1]:
            print(f"  going UP → peak is RIGHT")
            left = mid + 1
        else:
            print(f"  going DOWN → peak is LEFT (or AT mid)")
            right = mid

    print(f"  L=R={left} → peak at index {left}, value = {nums[left]}")
    return left


print("=== Find Peak Element (LC #162) ===")

test_cases = [
    [1, 2, 3, 1],
    [1, 2, 1, 3, 5, 6, 4],
    [3, 2, 1],
    [1, 2, 3, 4, 5],
]

for nums in test_cases:
    print(f"\nArray: {nums}")
    idx = find_peak_element(nums)
    print(f"→ Peak index: {idx} (value: {nums[idx]})")

---

## Problem 5: Search a 2D Matrix (LC #74)

Search a 2D matrix where:
- Each row is sorted left to right
- First element of each row > last element of previous row

Treat the entire matrix as one sorted 1D array.

```
Matrix:              Flattened (conceptual):
┌────┬────┬────┬────┐
│  1 │  3 │  5 │  7 │   [1, 3, 5, 7, 10, 11, 16, 20, 23, 30, 34, 60]
├────┼────┼────┼────┤    0  1  2  3   4   5   6   7   8   9  10  11
│ 10 │ 11 │ 16 │ 20 │
├────┼────┼────┼────┤
│ 23 │ 30 │ 34 │ 60 │
└────┴────┴────┴────┘

3 rows × 4 cols = 12 elements

Convert 1D index → 2D:
  row = idx // cols
  col = idx % cols

Example: idx=7 in 3×4 matrix
  row = 7 // 4 = 1
  col = 7 % 4  = 3  → matrix[1][3] = 20 ✓
```

In [ ]:
def search_matrix(matrix, target):
    if not matrix or not matrix[0]:
        return False

    rows, cols = len(matrix), len(matrix[0])
    left, right = 0, rows * cols - 1

    while left <= right:
        mid = left + (right - left) // 2
        row, col = mid // cols, mid % cols
        val = matrix[row][col]
        print(f"  L={left} R={right} mid_idx={mid} → [{row}][{col}] = {val}", end="")

        if val == target:
            print(f"  ✓ FOUND")
            return True
        elif val < target:
            print(f"  < {target} → go right")
            left = mid + 1
        else:
            print(f"  > {target} → go left")
            right = mid - 1

    print(f"  NOT FOUND")
    return False


print("=== Search a 2D Matrix (LC #74) ===")

matrix = [
    [1,  3,  5,  7],
    [10, 11, 16, 20],
    [23, 30, 34, 60]
]

print("Matrix:")
for row in matrix:
    print(f"  {row}")

for target in [3, 16, 60, 13]:
    print(f"\nSearch for {target}:")
    result = search_matrix(matrix, target)
    print(f"→ Found: {result}")

---

## Binary Search on Answer (The Advanced Pattern)

This is the **most powerful** binary search pattern, and often the hardest to recognize.

Instead of searching **in the data**, you search on the **answer space**.

```
┌─────────────────────────────────────────────────────────────────────┐
│                  BINARY SEARCH ON ANSWER                           │
│                                                                     │
│  Classic BS:    "Is the target at index mid?"                       │
│  BS on Answer:  "Can we achieve the goal with answer = mid?"        │
│                                                                     │
│  Instead of searching through data, we search through              │
│  POSSIBLE ANSWERS and check if each is feasible.                   │
└─────────────────────────────────────────────────────────────────────┘
```

### How It Works

**Example**: "What's the minimum ship capacity to deliver all packages in D days?"

```
weights = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],  days = 5

The answer must be between:
  lo = max(weights) = 10   (must carry the heaviest package)
  hi = sum(weights) = 55   (carry everything in 1 day)

Search space of ANSWERS: [10, 11, 12, ..., 55]
                          lo                 hi

For each candidate capacity, ASK: "Can we ship in ≤ 5 days?"

  capacity=32 → 2 days  ✓ feasible → try smaller (hi = mid)
  capacity=21 → 3 days  ✓ feasible → try smaller (hi = mid)
  capacity=15 → 5 days  ✓ feasible → try smaller (hi = mid)
  capacity=12 → 6 days  ✗ too many → need bigger  (lo = mid+1)
  capacity=13 → 6 days  ✗ too many → need bigger  (lo = mid+1)
  capacity=14 → 5 days  ✓ feasible → try smaller (hi = mid)
  lo == hi == 15  → answer is 15!
```

### The Template

```python
lo, hi = min_possible, max_possible
while lo < hi:
    mid = (lo + hi) // 2
    if feasible(mid):    # can we achieve the goal with this answer?
        hi = mid         # yes → try a smaller/better answer
    else:
        lo = mid + 1     # no → need a bigger answer
return lo                # lo == hi == minimum feasible answer
```

### When to Recognize This Pattern

```
  Trigger phrases in problem descriptions:

  ┌────────────────────────────────────────────────────────┐
  │ "Minimize the maximum ..."                             │
  │ "Maximize the minimum ..."                             │
  │ "Find the minimum capacity/speed/time such that ..."   │
  │ "What is the least X needed to ..."                    │
  │ "Can you do it within X?"                              │
  └────────────────────────────────────────────────────────┘

  All require two things:
  1. A RANGE of possible answers (lo to hi)
  2. A FEASIBILITY CHECK (monotonic: if X works, X+1 also works)
```

---

## Problem 6: Koko Eating Bananas (LC #875)

Koko eats bananas at speed `k` per hour. Each pile takes `ceil(pile/k)` hours.  
Find the **minimum** `k` so she finishes all piles in `h` hours.

```
piles = [3, 6, 7, 11], h = 8

Answer range: [1, max(piles)] = [1, 11]

Feasibility check:
  k=6: ceil(3/6) + ceil(6/6) + ceil(7/6) + ceil(11/6)
     = 1 + 1 + 2 + 2 = 6 hours ≤ 8  ✓
  k=3: ceil(3/3) + ceil(6/3) + ceil(7/3) + ceil(11/3)
     = 1 + 2 + 3 + 4 = 10 hours > 8  ✗
  k=4: 1 + 2 + 2 + 3 = 8 hours ≤ 8  ✓  ← minimum!
```

In [ ]:
import math

def min_eating_speed(piles, h):
    def feasible(k):
        hours = sum(math.ceil(p / k) for p in piles)
        return hours <= h

    lo, hi = 1, max(piles)
    print(f"  Answer range: [{lo}, {hi}]")

    while lo < hi:
        mid = (lo + hi) // 2
        hours = sum(math.ceil(p / mid) for p in piles)
        ok = hours <= h
        print(f"  k={mid}: hours={hours} {'≤' if ok else '>'} {h}  {'✓ try smaller' if ok else '✗ need bigger'}")
        if ok:
            hi = mid
        else:
            lo = mid + 1

    print(f"  → Minimum speed: {lo}")
    return lo


print("=== Koko Eating Bananas (LC #875) ===")

print(f"\npiles=[3,6,7,11], h=8")
min_eating_speed([3, 6, 7, 11], 8)

print(f"\npiles=[30,11,23,4,20], h=5")
min_eating_speed([30, 11, 23, 4, 20], 5)

print(f"\npiles=[30,11,23,4,20], h=6")
min_eating_speed([30, 11, 23, 4, 20], 6)

---

## Problem 7: Capacity to Ship Packages Within D Days (LC #1011)

Ship packages on a conveyor belt. Each day, load packages **in order** until capacity is reached.  
Find the **minimum** ship capacity to ship all packages within `days` days.

```
weights = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], days = 5

Answer range:
  lo = max(weights) = 10  (must fit the heaviest)
  hi = sum(weights) = 55  (ship everything day 1)

Feasibility: greedily fill each day until capacity hit, count days.
  capacity=15:
    Day 1: [1,2,3,4,5] = 15  Day 2: [6,7] = 13
    Day 3: [8] = 8           Day 4: [9] = 9
    Day 5: [10] = 10         → 5 days ✓
```

In [ ]:
def ship_within_days(weights, days):
    def feasible(capacity):
        day_count = 1
        current_load = 0
        for w in weights:
            if current_load + w > capacity:
                day_count += 1
                current_load = 0
            current_load += w
        return day_count <= days

    lo, hi = max(weights), sum(weights)
    print(f"  Answer range: [{lo}, {hi}]")

    while lo < hi:
        mid = (lo + hi) // 2
        day_count = 1
        current = 0
        shipments = []
        current_ship = []
        for w in weights:
            if current + w > mid:
                shipments.append(current_ship)
                day_count += 1
                current = 0
                current_ship = []
            current += w
            current_ship.append(w)
        shipments.append(current_ship)
        ok = day_count <= days
        print(f"  cap={mid}: {day_count} days {shipments} {'✓' if ok else '✗'}")
        if ok:
            hi = mid
        else:
            lo = mid + 1

    print(f"  → Minimum capacity: {lo}")
    return lo


print("=== Capacity to Ship Packages (LC #1011) ===")

print(f"\nweights=[1,2,3,4,5,6,7,8,9,10], days=5")
ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5)

print(f"\nweights=[3,2,2,4,1,4], days=3")
ship_within_days([3, 2, 2, 4, 1, 4], 3)

print(f"\nweights=[1,2,3,1,1], days=4")
ship_within_days([1, 2, 3, 1, 1], 4)

---

## Problem 8: Split Array Largest Sum (LC #410)

Split array into `k` non-empty contiguous subarrays.  
**Minimize** the largest sum among the subarrays.

```
nums = [7, 2, 5, 10, 8], k = 2

Possible splits:
  [7]       | [2,5,10,8]  → max(7, 25) = 25
  [7,2]     | [5,10,8]    → max(9, 23) = 23
  [7,2,5]   | [10,8]      → max(14, 18) = 18  ← minimum!
  [7,2,5,10]| [8]         → max(24, 8) = 24

Binary search on answer:
  lo = max(nums) = 10   (each subarray needs at least the max element)
  hi = sum(nums) = 32   (one subarray holds everything)

  Feasibility: "Can we split into ≤ k parts where each part ≤ mid?"
  Greedily fill subarrays until sum exceeds mid, start a new one.
```

In [ ]:
def split_array(nums, k):
    def can_split(max_sum):
        parts = 1
        current = 0
        for n in nums:
            if current + n > max_sum:
                parts += 1
                current = 0
            current += n
        return parts <= k

    lo, hi = max(nums), sum(nums)
    print(f"  Answer range: [{lo}, {hi}]")

    while lo < hi:
        mid = (lo + hi) // 2
        parts = 1
        current = 0
        groups = []
        group = []
        for n in nums:
            if current + n > mid:
                groups.append(group)
                parts += 1
                current = 0
                group = []
            current += n
            group.append(n)
        groups.append(group)
        ok = parts <= k
        sums = [sum(g) for g in groups]
        print(f"  max_sum={mid}: {groups} sums={sums} parts={parts} {'✓' if ok else '✗'}")
        if ok:
            hi = mid
        else:
            lo = mid + 1

    print(f"  → Minimum largest sum: {lo}")
    return lo


print("=== Split Array Largest Sum (LC #410) ===")

print(f"\nnums=[7,2,5,10,8], k=2")
split_array([7, 2, 5, 10, 8], 2)

print(f"\nnums=[1,2,3,4,5], k=2")
split_array([1, 2, 3, 4, 5], 2)

print(f"\nnums=[1,4,4], k=3")
split_array([1, 4, 4], 3)

---

## Practice Problems

| # | Problem | Difficulty | Pattern | LC # |
|---|---------|-----------|---------|------|
| 1 | Binary Search | Easy | Standard template | 704 |
| 2 | Search Insert Position | Easy | Find boundary | 35 |
| 3 | Find First and Last Position | Medium | Boundary tracking | 34 |
| 4 | Search in Rotated Sorted Array | Medium | Check sorted half | 33 |
| 5 | Find Minimum in Rotated Sorted Array | Medium | Compare mid vs right | 153 |
| 6 | Find Peak Element | Medium | Compare neighbors | 162 |
| 7 | Search a 2D Matrix | Medium | 2D → 1D mapping | 74 |
| 8 | Koko Eating Bananas | Medium | BS on answer | 875 |
| 9 | Capacity to Ship Packages | Medium | BS on answer | 1011 |
| 10 | Split Array Largest Sum | Hard | BS on answer | 410 |
| 11 | Find Minimum in Rotated (with duplicates) | Hard | Modified BS | 154 |
| 12 | Median of Two Sorted Arrays | Hard | BS on partition | 4 |
| 13 | Magnetic Force Between Two Balls | Medium | BS on answer | 1552 |
| 14 | Minimum Number of Days to Make m Bouquets | Medium | BS on answer | 1482 |

---

## Pattern Cheat Sheet

```
┌────────────────────────────────────────────────────────────────────────┐
│              BINARY SEARCH — PATTERN RECOGNITION                     │
├────────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  "Find exact value in sorted"                                        │
│       → Standard binary search (Template 1: left <= right)           │
│                                                                      │
│  "Find first/last occurrence"                                        │
│       → Binary search with boundary tracking (don't return early)    │
│                                                                      │
│  "Sorted but rotated"                                                │
│       → Check which half is sorted, decide accordingly               │
│                                                                      │
│  "Find peak/valley"                                                  │
│       → Compare mid with neighbors (going up → peak is right)        │
│                                                                      │
│  "Minimize the maximum"                                              │
│       → Binary search on answer + feasibility check                  │
│                                                                      │
│  "Find minimum capacity/speed"                                       │
│       → Binary search on answer + feasibility check                  │
│                                                                      │
├────────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  COMMON BUGS:                                                        │
│  • Infinite loop:  check that lo or hi always changes                │
│  • Off-by-one:     verify which template you're using                │
│  • Overflow:       use lo + (hi - lo) // 2 instead of (lo + hi) // 2 │
│  • Wrong half:     draw it out, trace with examples                  │
│                                                                      │
│  COMPLEXITY (all variants):                                          │
│  • Time:  O(log n)  — or O(log n × check) for BS on answer          │
│  • Space: O(1)                                                       │
│                                                                      │
└────────────────────────────────────────────────────────────────────────┘
```

---

**Next up: Topic 9 — Trees**